# Future Sales Prediction - Kaggle Competition

### Modeling File

https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/

#### File descriptions
- sales_train.csv - the training set. Daily historical data from January 2013 to October 2015.

- test.csv - the test set. You need to forecast the sales for these shops and products for November 2015.

- sample_submission.csv - a sample submission file in the correct format.

- items.csv - supplemental information about the items/products.

- item_categories.csv  - supplemental information about the items categories.

- shops.csv- supplemental information about the shops.

#### Data fields

- ID - an Id that represents a (Shop, Item) tuple within the test set

- shop_id - unique identifier of a shop

- item_id - unique identifier of a product

- item_category_id - unique identifier of item category

- item_cnt_day - number of products sold. You are predicting a monthly amount of this measure

- item_price - current price of an item

- date - date in format dd/mm/yyyy

- date_block_num - a consecutive month number, used for convenience. January 2013 is 0, February 2013 is 1,..., October 2015 is 33

- item_name - name of item

- shop_name - name of shop

- item_category_name - name of item category

### I. Import the dataset and data analysis libraries

In [86]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [87]:
sales_train = pd.read_csv("data/sales_train.csv")
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


In [88]:
test = pd.read_csv("data/test.csv")
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


In [89]:
sample_submission = pd.read_csv("data/sample_submission.csv")
print(sample_submission.shape)
sample_submission.head()

(214200, 2)


,ID,item_cnt_month
0,0,0.5
1,1,0.5
2,2,0.5
3,3,0.5
4,4,0.5


In [90]:
items = pd.read_csv("data/items.csv")
items.head()

,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


In [91]:
item_categories = pd.read_csv("data/item_categories.csv")
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


In [92]:
shops = pd.read_csv("data/shops.csv")
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


### II. Initial data checks and manipulations

In [93]:
sales_train['date'] = pd.to_datetime(sales_train['date'], format = '%d.%m.%Y')
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,2013-01-02,0,59,22154,999.00,1.0
1,2013-01-03,0,25,2552,899.00,1.0
2,2013-01-05,0,25,2552,899.00,-1.0
3,2013-01-06,0,25,2554,1709.05,1.0
4,2013-01-15,0,25,2555,1099.00,1.0


In [94]:
# Split into month, day, and year
sales_train['year'] = sales_train['date'].dt.year
sales_train['month'] = sales_train['date'].dt.month
sales_train['day'] = sales_train['date'].dt.day
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3
2,2013-01-05,0,25,2552,899.00,-1.0,2013,1,5
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15


In [95]:
sales_train.isna().sum()

date              0
date_block_num    0
shop_id           0
item_id           0
item_price        0
item_cnt_day      0
year              0
month             0
day               0
dtype: int64

In [96]:
sales_train.nunique()

date               1034
date_block_num       34
shop_id              60
item_id           21807
item_price        19993
item_cnt_day        198
year                  3
month                12
day                  31
dtype: int64

In [97]:
# Describing datetime and quantitative variables
sales_train[['date', 'item_price', 'item_cnt_day']].describe()

,date,item_price,item_cnt_day
count,2935849,2.935849e+06,2.935849e+06
mean,2014-04-03 05:44:34.970681344,8.908532e+02,1.242641e+00
min,2013-01-01 00:00:00,-1.000000e+00,-2.200000e+01
25%,2013-08-01 00:00:00,2.490000e+02,1.000000e+00
50%,2014-03-04 00:00:00,3.990000e+02,1.000000e+00
75%,2014-12-05 00:00:00,9.990000e+02,1.000000e+00
max,2015-10-31 00:00:00,3.079800e+05,2.169000e+03
std,NaN,1.729800e+03,2.618834e+00


In [98]:
# Declaring categorical variables
sales_train['shop_id'] = pd.Categorical(sales_train['shop_id'])
sales_train['item_id'] = pd.Categorical(sales_train['item_id'])

In [99]:
# Convert negative to positive values in item_cnt_day
sales_train['item_cnt_day'] = sales_train['item_cnt_day'].abs()

In [100]:
# Create a "master table" for EDA in training data
sales_full = sales_train.merge(items, how = 'inner', on = 'item_id')\
                              .merge(item_categories, how = 'inner', on = 'item_category_id')\
                              .merge(shops, how = 'inner', on = 'shop_id')

sales_full.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,item_name,item_category_id,item_category_name,shop_name
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,ЯВЛЕНИЕ 2012 (BD),37,Кино - Blu-Ray,"Ярославль ТЦ ""Альтаир"""
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
2,2013-01-05,0,25,2552,899.00,1.0,2013,1,5,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,DEEP PURPLE Who Do You Think We Are LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,DEEP PURPLE 30 Very Best Of 2CD (Фирм.),56,Музыка - CD фирменного производства,"Москва ТРК ""Атриум"""


#### Data checks:

1. Is there any item sold at more than one different price?

In [101]:
shops_and_items = sales_full[['shop_id', 'item_id', 'item_price']].copy()
shops_and_items.drop_duplicates(['shop_id', 'item_id'], inplace = True)

# Same shop, same item, different prices?
print(len(shops_and_items[shops_and_items.duplicated(subset = ['shop_id', 'item_id', 'item_price'])]))

0


2. Is there any group of items having different IDs but the same name? Do the same with shop name and item category name.

In [102]:
print(items.duplicated(['item_id']).sum()) # Number of duplicated item IDs
print(items.duplicated(['item_name']).sum()) # Number of duplicated item names

0
0


In [103]:
# With item categories
print(item_categories.duplicated(['item_category_id']).sum())
print(item_categories.duplicated(['item_category_name']).sum())

0
0


In [104]:
# With shops
print(shops.duplicated(['shop_id']).sum())
print(shops.duplicated(['shop_name']).sum())

0
0


3. Is there any day when the shop can't sell any item?

In [123]:
# Generate list of datetime indices - all days of sales in the training data
datetime_indices = sales_full[['date', 'date_block_num']].copy()
datetime_indices.drop_duplicates(inplace = True)
datetime_indices.sort_values('date', inplace = True)
datetime_indices.set_index('date', inplace = True)
print(datetime_indices.index)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-10-22', '2015-10-23', '2015-10-24', '2015-10-25',
               '2015-10-26', '2015-10-27', '2015-10-28', '2015-10-29',
               '2015-10-30', '2015-10-31'],
              dtype='datetime64[ns]', name='date', length=1034, freq=None)


In [121]:
# Generate a list of dates from the first and last day of sales in the training data
date_range = pd.date_range(start = sales_full['date'].min(), end = sales_full['date'].max())
print(date_range)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-10-22', '2015-10-23', '2015-10-24', '2015-10-25',
               '2015-10-26', '2015-10-27', '2015-10-28', '2015-10-29',
               '2015-10-30', '2015-10-31'],
              dtype='datetime64[ns]', length=1034, freq='D')


### III. Feature engineering

Aggregate existing features:

In [211]:
# Keep the necessary columns only - we are predicting by month
sales_features = sales_full[['year', 'month', 'shop_id', 'item_category_id', 'item_id', 'item_price', 'item_cnt_day']].copy()

# Because no item has more than 1 price, we can group item price by the median
sales_features_price = sales_features[['year', 'month', 'shop_id', 'item_category_id', 'item_id', 'item_price']]\
    .groupby(['year', 'month', 'shop_id', 'item_category_id', 'item_id'], as_index = False).median()

# Group item count by month by the sum
sales_features_itemcnt = sales_features[['year', 'month', 'shop_id', 'item_category_id', 'item_id', 'item_cnt_day']]\
    .groupby(['year', 'month', 'shop_id', 'item_category_id', 'item_id'], as_index = False).sum()

sales_features_itemcnt.rename(columns = {'item_cnt_day': 'item_cnt_month'}, inplace = True)

sales_features = sales_features_price.merge(sales_features_itemcnt, how = 'inner', on = ['year', 'month', 'shop_id', 'item_category_id', 'item_id'])
sales_features.sort_values(['year', 'month', 'shop_id', 'item_category_id', 'item_id', 'item_price', 'item_cnt_month'], inplace = True)
sales_features.head()

,year,month,shop_id,item_category_id,item_id,item_price,item_cnt_month
0,2013,1,0,2,5572,1322.0,10.0
1,2013,1,0,2,5573,560.0,1.0
2,2013,1,0,2,5575,806.0,4.0
3,2013,1,0,2,5576,2231.0,5.0
4,2013,1,0,2,5609,2381.0,1.0


Create weekend and holiday features:

In [251]:
# Separate the sales days together
sales_days = sales_full[['date']].copy()
sales_days.drop_duplicates(inplace = True, ignore_index = True)
sales_days.sort_values('date', inplace = True, ignore_index = True)

# Create month, year, and weekend indicator variables
sales_days['year'] = sales_days['date'].dt.year
sales_days['month'] = sales_days['date'].dt.month
sales_days['weekend_indicator'] = sales_days['date'].case_when(
    [(sales_days['date'].dt.day_of_week.isin([5, 6]), 1),
     (~sales_days['date'].dt.day_of_week.isin([5, 6]), 0)]
)

sales_days.head(7)

,date,year,month,weekend_indicator
0,2013-01-01,2013,1,0
1,2013-01-02,2013,1,0
2,2013-01-03,2013,1,0
3,2013-01-04,2013,1,0
4,2013-01-05,2013,1,1
5,2013-01-06,2013,1,1
6,2013-01-07,2013,1,0


In [252]:
# See Analysis File for more details
# List of holidays in Russia: https://www.timeanddate.com/holidays/russia/2013 (same as for 2014 and 2015)
holiday_list_2013 = ['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04', '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
                     '2013-02-23', '2013-03-08', '2013-05-01', '2013-05-02', '2013-05-03', '2013-05-09', '2013-05-10',
                     '2013-06-12', '2013-11-04']

holiday_list_2014 = ['2014-01-01', '2014-01-02', '2014-01-03', '2014-01-06', '2014-01-07', '2014-01-08',
                     '2014-02-22', '2014-02-23', '2014-03-08', '2014-03-09', '2014-03-10', '2014-05-01', '2014-05-02',
                     '2014-05-03', '2014-05-09', '2014-05-10', '2014-05-11', '2014-06-12', '2014-06-13', '2014-06-14', '2014-06-15',
                     '2014-11-01', '2014-11-02', '2014-11-03', '2014-11-04']

holiday_list_2015 = ['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04', '2015-01-05', '2015-01-06', '2015-01-07',
                     '2015-01-08', '2015-01-09', '2015-02-23', '2015-03-08', '2015-03-09', '2015-05-01', '2015-05-04', '2015-05-09',
                     '2015-05-11', '2015-06-12'] # only count until October 2015

# Combine to have a list of holidays in training data
holidays = holiday_list_2013 + holiday_list_2014 + holiday_list_2015
holidays = [pd.Timestamp(x) for x in holidays] # Convert the whole list to timestamp format

# Create holiday indicator variable
sales_days['holiday_indicator'] = sales_days['date'].case_when(
    [(sales_days['date'].isin(holidays), 1),
     (~sales_days['date'].isin(holidays), 0)]
)

sales_days.head()

,date,year,month,weekend_indicator,holiday_indicator
0,2013-01-01,2013,1,0,1
1,2013-01-02,2013,1,0,1
2,2013-01-03,2013,1,0,1
3,2013-01-04,2013,1,0,1
4,2013-01-05,2013,1,1,1


In [253]:
# Count the number of days as weekends and holidays in a (month, year) pair
sales_days.drop('date', axis = 1, inplace = True)
sales_days = sales_days.groupby(['year', 'month'], as_index = False).sum()
sales_days.rename(columns = {'weekend_indicator': 'weekends_in_month', 'holiday_indicator': 'holidays_in_month'}, inplace = True)
sales_days.head(7)

,year,month,weekends_in_month,holidays_in_month
0,2013,1,8,8
1,2013,2,8,1
2,2013,3,10,1
3,2013,4,8,0
4,2013,5,8,5
5,2013,6,10,1
6,2013,7,8,0


Create moving and lagged metrics (all shops and items, aggregated monthly):

In [255]:
sales_month = sales_features[['year', 'month', 'item_cnt_month']].copy()
sales_month = sales_month.groupby(['year', 'month'], as_index = False).sum()

# Lag features for 1 month and 3 months
sales_month['lag_1m'] = sales_month['item_cnt_month'].shift(periods = 1)
sales_month['lag_3m'] = sales_month['item_cnt_month'].shift(periods = 3)

# Percentage (fractional) change by 1 month and 3 months
sales_month['pct_change_1m'] = sales_month['item_cnt_month'].pct_change(periods = 1)
sales_month['pct_change_3m'] = sales_month['item_cnt_month'].pct_change(periods = 3)

# Moving average sales for the last 3 months
sales_month['moving_3m'] = sales_month['item_cnt_month'].rolling(window = 3).mean()

# Imputation by backfill for missing data
# Assume that sales data by month doesn't change before Jan 2013
sales_month.bfill(inplace = True)

# Rename a column further clarification
sales_month.rename(columns = {'item_cnt_month': 'total_monthly_quantity'}, inplace = True)

sales_month.head(7)

,year,month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m
0,2013,1,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667
1,2013,2,128674.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667
2,2013,3,147734.0,128674.0,132221.0,0.148126,-0.185545,136209.666667
3,2013,4,107688.0,147734.0,132221.0,-0.271068,-0.185545,128032.000000
4,2013,5,107326.0,107688.0,128674.0,-0.003362,-0.165908,120916.000000
5,2013,6,125785.0,107326.0,147734.0,0.171990,-0.148571,113599.666667
6,2013,7,117364.0,125785.0,107688.0,-0.066948,0.089852,116825.000000


Create seasonal features:

In [256]:
sales_month['season'] = sales_month['month'].case_when(
    [(sales_month['month'].isin([3, 4, 5]), 'spring'),
     (sales_month['month'].isin([6, 7, 8]), 'summer'),
     (sales_month['month'].isin([9, 10, 11]), 'fall'),
     (sales_month['month'].isin([12, 1, 2]), 'winter')]
)

# Because winter is very cold in Russia, I will encode so that winter goes first
sales_month['season'] = pd.Categorical(values = sales_month['season'], ordered = True,
                                       categories = ['winter', 'spring', 'summer', 'fall'])

# Use one-hot encoding to turn the seasons into features
# Drop the first column (winter) to avoid multicollinearity
onehot_season = pd.get_dummies(sales_month['season'], drop_first = True)

# Convert data type of all the one-hot encoded values: True/False to 0/1
onehot_season = onehot_season.map(lambda x: int(x))

# Merge the one-hot encoding into the sales_month feature DataFrame
sales_month = pd.concat([sales_month, onehot_season], axis = 1)

# Drop the categorical 'season' column now that we have 3 one-hot encoded season columns
sales_month.drop('season', axis = 1, inplace = True)

sales_month.head(7)

,year,month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m,spring,summer,fall
0,2013,1,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
1,2013,2,128674.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
2,2013,3,147734.0,128674.0,132221.0,0.148126,-0.185545,136209.666667,1,0,0
3,2013,4,107688.0,147734.0,132221.0,-0.271068,-0.185545,128032.000000,1,0,0
4,2013,5,107326.0,107688.0,128674.0,-0.003362,-0.165908,120916.000000,1,0,0
5,2013,6,125785.0,107326.0,147734.0,0.171990,-0.148571,113599.666667,0,1,0
6,2013,7,117364.0,125785.0,107688.0,-0.066948,0.089852,116825.000000,0,1,0


Now combine the features together:

In [258]:
final_features = sales_features.merge(sales_days, how = 'left', on = ['year', 'month'])\
    .merge(sales_month, how = 'left', on = ['year', 'month'])

final_features.head(7)

,year,month,shop_id,item_category_id,item_id,item_price,item_cnt_month,weekends_in_month,holidays_in_month,total_monthly_quantity,lag_1m,lag_3m,pct_change_1m,pct_change_3m,moving_3m,spring,summer,fall
0,2013,1,0,2,5572,1322.0,10.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
1,2013,1,0,2,5573,560.0,1.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
2,2013,1,0,2,5575,806.0,4.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
3,2013,1,0,2,5576,2231.0,5.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
4,2013,1,0,2,5609,2381.0,1.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
5,2013,1,0,2,5612,3623.0,1.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
6,2013,1,0,2,5623,294.0,1.0,8,8,132221.0,132221.0,132221.0,-0.026826,-0.185545,136209.666667,0,0,0
